# HeyDesk's speech-to-text on a free GPU (D87)

Our Bangla–English Whisper (`kayees/whisper-medium-bn-en-cs-faster`, faster-whisper) on a
Kaggle T4, for demos. `MODEL = "bengaliai"` serves BengaliAI's Bangla Whisper medium
(`bengaliAI/tugstugi_bengaliai-asr_whisper-medium`) instead, to compare: it's a Transformers
checkpoint, so the first run converts it to CTranslate2 (3 GB download, a few minutes). HeyDesk sends each utterance here when the caller stops talking
(`STT_PROVIDER=whisper`): `POST /transcribe?language=bn` with the WAV as the body, and gets
`{"text": …}` back.

**Run it on Kaggle:**

1. kaggle.com → **Create → New notebook → File → Import notebook**, and pick this file.
2. Right panel, **Settings**: *Accelerator* **GPU T4 x2**, *Internet* **on**
   (both need a phone-verified Kaggle account).
3. **Run all.** The first run downloads the model (775 MB) and takes 2 or 3 minutes.
4. The last cell transcribes a Bangla test clip and prints a line like
   `WHISPER_STT_URL=https://….trycloudflare.com`. Put it in HeyDesk's `.env` along with
   `STT_PROVIDER=whisper`, then restart HeyDesk.
5. Keep this tab open during the demo. Afterwards, **Run → Stop session** (Kaggle gives 30 GPU
   hours a week, shared with the TTS notebook if both run).

The URL changes on every run, and it is public while the session runs. Colab works too
technically, but Colab's terms forbid serving from a notebook. Use Kaggle.


In [ ]:
MODEL = "kayees"  # or "bengaliai", to compare (one per kernel: Run → Restart, then Run all)
MODELS = {
    "kayees": "kayees/whisper-medium-bn-en-cs-faster",  # ours: CTranslate2 int8, Bangla–English
    "bengaliai": "bengaliAI/tugstugi_bengaliai-asr_whisper-medium",  # Transformers fp32, Bangla
}
# For both models. [] is what our model's own Space uses (and bengaliai's generation config): it says faster-whisper's default list ([-1]) holds the
# space token in this model's Bengali tokenizer, so words run together. On 19 Oct, two TTS clips
# of "তিনশো ষাট টাকা" came out right with [-1], and as "30 টাকা" and "3000 টাকা" with []
SUPPRESS_TOKENS: list[int] = []
PORT = 8011

In [ ]:
import subprocess
import sys

subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "faster-whisper",
        "transformers",
        "nvidia-cublas-cu12",
        "nvidia-cudnn-cu12==9.*",
        "fastapi",
        "uvicorn",
    ],
    check=True,
)

In [ ]:
import ctypes
import glob
import io
import os
import subprocess
import wave

import ctranslate2
import numpy as np
import nvidia.cublas.lib
import nvidia.cudnn.lib
from faster_whisper import WhisperModel

# ctranslate2 is built for CUDA 12, and the notebook's own CUDA can be another version
# ("libcublas.so.12 is not found"): load the CUDA 12 libraries pip just installed, so it finds them.
for folder in (nvidia.cublas.lib, nvidia.cudnn.lib):
    for library in sorted(glob.glob(os.path.join(folder.__path__[0], "lib*.so*"))):
        ctypes.CDLL(library, mode=ctypes.RTLD_GLOBAL)

cuda = ctranslate2.get_cuda_device_count() > 0
assert cuda or os.environ.get("ALLOW_CPU"), "Settings → Accelerator: GPU T4 x2"
if MODEL == "kayees":
    path, precision = MODELS[MODEL], "int8_float16"
else:  # a Transformers checkpoint: faster-whisper needs it in CTranslate2, made once a session
    path, precision = "bengaliai-ct2", "float16"
    if not os.path.exists(path):
        subprocess.run(
            [
                "ct2-transformers-converter",
                *("--model", MODELS[MODEL], "--output_dir", path, "--quantization", "float16"),
                *("--copy_files", "tokenizer.json", "preprocessor_config.json"),
            ],
            check=True,
        )
whisper = WhisperModel(
    path, device="cuda" if cuda else "cpu", compute_type=precision if cuda else "int8"
)


def read_wav(data: bytes) -> np.ndarray:
    """HeyDesk sends 16 kHz mono 16-bit WAV, so we read it ourselves: faster-whisper's own
    reader (PyAV) breaks with PyAV 19."""
    with wave.open(io.BytesIO(data)) as wav:
        assert (wav.getframerate(), wav.getnchannels(), wav.getsampwidth()) == (16000, 1, 2)
        return np.frombuffer(wav.readframes(wav.getnframes()), "<i2").astype(np.float32) / 32768


def transcribe(audio: np.ndarray, language: str) -> str:
    """Decoded as the model's own Space does (bangal-english_asr/bn-en-asr-space/app.py)."""
    segments, _ = whisper.transcribe(
        audio,
        language=language,
        beam_size=1,
        temperature=0,
        without_timestamps=True,
        vad_filter=False,
        condition_on_previous_text=False,
        suppress_tokens=SUPPRESS_TOKENS,  # see the first cell
    )
    return " ".join(segment.text.strip() for segment in segments).strip()


transcribe(np.zeros(16000, dtype=np.float32), "bn")  # warm up: the first one is always slow
print("model ready")

In [ ]:
import asyncio
import threading
import time
from typing import Literal

import uvicorn
from fastapi import FastAPI, Request

app = FastAPI()
lock = asyncio.Lock()  # one utterance at a time


@app.post("/transcribe")
async def transcribe_route(
    request: Request, language: Literal["bn", "en"] = "bn"
) -> dict[str, str]:
    """The body is the utterance as a WAV file."""
    audio = read_wav(await request.body())
    async with lock:
        start = time.perf_counter()
        text = await asyncio.to_thread(transcribe, audio, language)
    print(f"an utterance in {time.perf_counter() - start:.2f} s")
    return {"text": text}


threading.Thread(
    target=uvicorn.run,
    args=(app,),
    kwargs={"host": "127.0.0.1", "port": PORT, "log_level": "warning"},
    daemon=True,
).start()

In [ ]:
import re
import subprocess
import urllib.request
from pathlib import Path

# A Cloudflare quick tunnel: a public https URL for this notebook's port, no account needed.
if not os.path.exists("cloudflared"):
    urllib.request.urlretrieve(
        "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64",
        "cloudflared",
    )
    os.chmod("cloudflared", 0o755)
log = open("cloudflared.log", "w")  # noqa: SIM115 (the tunnel writes to it while it runs)
tunnel = subprocess.Popen(
    ["./cloudflared", "tunnel", "--url", f"http://127.0.0.1:{PORT}", "--no-autoupdate"],
    stdout=log,
    stderr=log,
)
URL = None
for _ in range(60):
    found = re.search(
        r"https://[a-z0-9-]+\.trycloudflare\.com", Path("cloudflared.log").read_text()
    )
    if found:
        URL = found.group(0)
        break
    time.sleep(1)
assert URL, "no tunnel URL yet: see cloudflared.log"

In [ ]:
import json

from huggingface_hub import hf_hub_download
from IPython.display import Audio, display

# A Bangla clip, sent the way HeyDesk will send it. To the server here, not the tunnel:
# some notebooks can't reach a trycloudflare address from inside ("Network is unreachable").
clip = hf_hub_download("Banglabox/chatterbox-bangla-tts", "voices/ref_male.wav")
display(Audio(clip))
with wave.open(clip) as wav:  # 24 kHz mono 16-bit: made 16 kHz, as HeyDesk sends it
    samples = np.frombuffer(wav.readframes(wav.getnframes()), "<i2")
    rate = wav.getframerate()
samples = np.interp(np.arange(0, len(samples), rate / 16000), np.arange(len(samples)), samples)
body = io.BytesIO()
with wave.open(body, "wb") as wav:
    wav.setnchannels(1)
    wav.setsampwidth(2)
    wav.setframerate(16000)
    wav.writeframes(samples.astype("<i2").tobytes())
request = urllib.request.Request(
    f"http://127.0.0.1:{PORT}/transcribe?language=bn",
    data=body.getvalue(),
    headers={"Content-Type": "audio/wav"},
)
start = time.perf_counter()
with urllib.request.urlopen(request, timeout=60) as response:
    text = json.loads(response.read())["text"]
print(f"on this machine in {time.perf_counter() - start:.2f} s:", text)
print("\nPut these lines in HeyDesk's .env, then restart HeyDesk:\n")
print("STT_PROVIDER=whisper")
print(f"WHISPER_STT_URL={URL}")